# Activation Functions — Exercises

10 exercises covering the mathematics of activation functions, from basic derivatives through vanishing gradients, initialisation theory, and LLM architecture design.

| Format | Description |
|---|---|
| **Problem** | Markdown cell with task description |
| **Your Solution** | Code cell with scaffolding |
| **Solution** | Code cell with reference solution and checks |

### Difficulty Levels

| Level | Exercises | Focus |
|---|---|---|
| ★ | 1–3 | Core derivatives and basic properties |
| ★★ | 4–7 | Deeper theory: gradients, Lipschitz, initialisation |
| ★★★ | 8–10 | AI applications: SwiGLU, calibration, activation engineering |

### Topic Map

| Topic | Exercise |
|---|---|
| Sigmoid/tanh derivatives | 1 |
| ReLU gradient and GELU comparison | 2 |
| Softmax Jacobian | 3 |
| Vanishing gradient depth analysis | 4 |
| SELU self-normalising property | 5 |
| Lipschitz constants | 6 |
| He initialisation derivation | 7 |
| SwiGLU parameter budget | 8 |
| Temperature scaling calibration | 9 |
| Activation engineering toy model | 10 |


In [ ]:
import numpy as np
import scipy.special as scs
from scipy.optimize import minimize_scalar, brentq

try:
    import matplotlib.pyplot as plt
    HAS_MPL = True
except ImportError:
    HAS_MPL = False

np.set_printoptions(precision=6, suppress=True)
np.random.seed(42)

def header(title):
    print('\n' + '=' * len(title))
    print(title)
    print('=' * len(title))

def check_close(name, got, expected, tol=1e-6):
    ok = np.allclose(got, expected, atol=tol, rtol=tol)
    print(f"{'PASS' if ok else 'FAIL'} — {name}")
    if not ok:
        print(f'  expected: {expected}')
        print(f'  got:      {got}')
    return ok

def check_true(name, cond):
    print(f"{'PASS' if cond else 'FAIL'} — {name}")
    return cond

def sigmoid(z):
    return np.where(z >= 0, 1/(1+np.exp(-z)), np.exp(z)/(1+np.exp(z)))

def gelu(z):
    return 0.5 * z * (1 + scs.erf(z / np.sqrt(2)))

def silu(z):
    return z * sigmoid(z)

def relu(z):
    return np.maximum(0, z)

def elu(z, a=1.0):
    return np.where(z >= 0, z, a * (np.exp(np.clip(z, -20, 20)) - 1))

SELU_L = 1.0507009873554805
SELU_A = 1.6732632423543772
def selu(z):
    return SELU_L * elu(z, SELU_A)

def softmax(z, axis=-1):
    e = np.exp(z - z.max(axis=axis, keepdims=True))
    return e / e.sum(axis=axis, keepdims=True)

def log_softmax(z, axis=-1):
    m = z.max(axis=axis, keepdims=True)
    return z - m - np.log(np.sum(np.exp(z - m), axis=axis, keepdims=True))

print('Setup complete. Functions available: sigmoid, tanh (np.tanh), relu, elu, selu, gelu, silu, softmax, log_softmax')


---

## Exercise 1 ★ — Sigmoid and Tanh Derivatives

**Mathematical background:** The sigmoid derivative is $\sigma'(z) = \sigma(z)(1-\sigma(z))$.
The tanh derivative is $\tanh'(z) = 1 - \tanh^2(z) = \operatorname{sech}^2(z)$.

**Tasks:**
(a) Verify the sigmoid derivative formula: compute $\sigma'(z)$ both as $\sigma(z)(1-\sigma(z))$ and via numerical central difference, and confirm they match.

(b) Verify that $\tanh(z) = 2\sigma(2z) - 1$ for all test values.

(c) Find the value of $z$ that maximises $\sigma'(z)$. What is the maximum value?

(d) Find the value of $z$ that maximises $|\tanh'(z)|$. What is the maximum value?


In [ ]:
# Exercise 1: Your Solution

z_test = np.linspace(-5, 5, 1000)

# (a) Sigmoid derivative: analytic vs numerical
sig_analytic = None  # TODO: compute sigma(z) * (1 - sigma(z))
h = 1e-6
sig_numeric  = None  # TODO: central difference of sigmoid(z)

# (b) Tanh identity
tanh_direct   = np.tanh(z_test)
tanh_from_sig = None  # TODO: 2*sigmoid(2*z) - 1

# (c) Max of sigma'
z_max_sig = None  # TODO: z_test[argmax of sig_analytic]
max_sig_deriv = None  # TODO: value at that z

# (d) Max of |tanh'|
tanh_deriv = None  # TODO: 1 - tanh(z)^2
z_max_tanh = None  # TODO: z_test[argmax of tanh_deriv]
max_tanh_deriv = None  # TODO: value at that z

print('(a) Max |analytic - numeric|:', None)
print('(b) Max |tanh_direct - tanh_from_sig|:', None)
print(f'(c) Max sigma\' at z={z_max_sig}, value={max_sig_deriv}')
print(f'(d) Max |tanh\'| at z={z_max_tanh}, value={max_tanh_deriv}')


In [ ]:
# Exercise 1: Solution

z_test = np.linspace(-5, 5, 10000)
h = 1e-7

# (a) Sigmoid derivative
sig_vals     = sigmoid(z_test)
sig_analytic = sig_vals * (1 - sig_vals)
sig_numeric  = (sigmoid(z_test + h) - sigmoid(z_test - h)) / (2 * h)

# (b) Tanh identity
tanh_direct   = np.tanh(z_test)
tanh_from_sig = 2 * sigmoid(2 * z_test) - 1

# (c) Max of sigma'
idx_max_sig  = np.argmax(sig_analytic)
z_max_sig    = z_test[idx_max_sig]
max_sig_val  = sig_analytic[idx_max_sig]

# (d) Max of |tanh'|
tanh_deriv    = 1 - np.tanh(z_test)**2
idx_max_tanh  = np.argmax(tanh_deriv)
z_max_tanh    = z_test[idx_max_tanh]
max_tanh_val  = tanh_deriv[idx_max_tanh]

header('Exercise 1: Sigmoid and Tanh Derivatives')
check_close('sigma_analytic matches sigma_numeric',
            np.max(np.abs(sig_analytic - sig_numeric)), 0, tol=1e-5)
check_close('tanh(z) = 2*sigma(2z) - 1',
            np.max(np.abs(tanh_direct - tanh_from_sig)), 0, tol=1e-12)
check_close('max sigma prime = 1/4', max_sig_val, 0.25, tol=1e-4)
check_close('argmax sigma prime = 0', z_max_sig, 0.0, tol=0.01)
check_close('max |tanh prime| = 1', max_tanh_val, 1.0, tol=1e-4)
check_close('argmax tanh prime = 0', z_max_tanh, 0.0, tol=0.01)

print('\nTakeaway: Sigmoid max gradient = 1/4 (causes 4x attenuation per layer); '
      'tanh max gradient = 1 (no attenuation at origin, but saturates in tails).')


---

## Exercise 2 ★ — ReLU Gradient and GELU Comparison

**Mathematical background:**
$\operatorname{ReLU}'(z) = \mathbf{1}[z > 0]$ (Heaviside step function).
$\operatorname{GELU}'(z) = \Phi(z) + z\phi(z)$ where $\Phi$ is the standard normal CDF, $\phi$ is the PDF.

**Tasks:**
(a) Compute $\operatorname{ReLU}'(z)$ and $\operatorname{GELU}'(z)$ at $z \in \{-3, -1, 0, 1, 3\}$.

(b) Verify $\operatorname{GELU}'(0) = 0.5$ analytically.

(c) Find the minimum of GELU (approximately). What is $\operatorname{GELU}(-0.751)$?

(d) Show that $\operatorname{GELU}(z) \approx z$ for large $z$ (i.e., the error $|\operatorname{GELU}(z) - z|$ is small).


In [ ]:
# Exercise 2: Your Solution

z_pts = np.array([-3.0, -1.0, 0.0, 1.0, 3.0])

# (a) Derivatives at test points
relu_deriv_pts = None  # TODO: ReLU'(z) = 1[z > 0]
gelu_deriv_pts = None  # TODO: Phi(z) + z*phi(z) where Phi=norm CDF, phi=norm PDF

# (b) GELU'(0) = 0.5
gelu_deriv_at_0 = None  # TODO: evaluate GELU' at z=0

# (c) GELU minimum
gelu_at_minus_0751 = None  # TODO: gelu(-0.751)

# (d) GELU(z) approx z for large z
z_large = np.array([5.0, 10.0, 20.0])
gelu_large = None  # TODO
error_large = None  # TODO: |gelu(z) - z|

print('(a) ReLU derivatives:', relu_deriv_pts)
print('(a) GELU derivatives:', gelu_deriv_pts)
print('(b) GELU\'(0):', gelu_deriv_at_0)
print('(c) GELU(-0.751):', gelu_at_minus_0751)
print('(d) |GELU(z) - z| for large z:', error_large)


In [ ]:
# Exercise 2: Solution
from scipy.stats import norm

z_pts = np.array([-3.0, -1.0, 0.0, 1.0, 3.0])

# (a) ReLU and GELU derivatives
relu_deriv_pts = (z_pts > 0).astype(float)
Phi = scs.ndtr  # standard normal CDF
phi = lambda z: np.exp(-0.5*z**2) / np.sqrt(2*np.pi)  # standard normal PDF
gelu_deriv_pts = Phi(z_pts) + z_pts * phi(z_pts)

# (b) GELU'(0)
gelu_deriv_at_0 = float(Phi(0.0) + 0.0 * phi(0.0))  # = 0.5 + 0 = 0.5

# (c) GELU minimum
gelu_at_minus_0751 = float(gelu(np.array([-0.751])))
# Exact minimum location
res = minimize_scalar(gelu, bounds=(-2, 0), method='bounded')

# (d) GELU approx z for large z
z_large = np.array([5.0, 10.0, 20.0])
gelu_large = gelu(z_large)
error_large = np.abs(gelu_large - z_large)

header('Exercise 2: ReLU Gradient and GELU Comparison')
print(f'ReLU derivatives at {z_pts}: {relu_deriv_pts}')
print(f'GELU derivatives at {z_pts}: {np.round(gelu_deriv_pts, 4)}')
check_close("GELU'(0) = 0.5", gelu_deriv_at_0, 0.5, tol=1e-10)
check_close('GELU(-0.751) ≈ -0.170', gelu_at_minus_0751, -0.170, tol=0.005)
check_close('GELU minimum ≈ -0.170', res.fun, -0.170, tol=0.005)
check_true('|GELU(z) - z| < 0.01 for z=20', error_large[-1] < 0.01)

print('\nTakeaway: GELU is nearly linear for large z (gradient ≈ 1), '
      'slightly suppressed near origin (gradient = 0.5), '
      'and has a soft negative regime (minimum ≈ -0.170).')


---

## Exercise 3 ★ — Softmax Jacobian

**Mathematical background:** For $\mathbf{s} = \operatorname{softmax}(\mathbf{z})$:
$$J_{ij} = \frac{\partial s_i}{\partial z_j} = s_i(\delta_{ij} - s_j) \quad \Rightarrow \quad J = \operatorname{diag}(\mathbf{s}) - \mathbf{s}\mathbf{s}^\top$$

**Tasks:**
(a) Given $\mathbf{z} = [2, 1, 0.5, -1]$, compute $\mathbf{s} = \operatorname{softmax}(\mathbf{z})$.

(b) Construct the analytic Jacobian $J = \operatorname{diag}(\mathbf{s}) - \mathbf{s}\mathbf{s}^\top$.

(c) Verify $J$ against numerical central differences.

(d) Verify that $J \mathbf{1} = \mathbf{0}$ (shift invariance). Explain why.

(e) Using $J$, derive the cross-entropy gradient: show $\nabla_{\mathbf{z}} (-\log s_y) = \mathbf{s} - \mathbf{e}_y$.


In [ ]:
# Exercise 3: Your Solution

z = np.array([2.0, 1.0, 0.5, -1.0])
y = 0  # true class index

# (a) Softmax
s = None  # TODO: softmax(z)

# (b) Jacobian
J = None  # TODO: np.diag(s) - np.outer(s, s)

# (c) Numerical Jacobian
h = 1e-6
K = len(z)
J_num = np.zeros((K, K))
for j in range(K):
    pass  # TODO: fill J_num[:, j] via central differences

# (d) Shift invariance
shift_test = None  # TODO: J @ np.ones(K)

# (e) Cross-entropy gradient
e_y = np.zeros(K); e_y[y] = 1.0
grad_ce_analytic = None  # TODO: s - e_y

print('(a) s =', s)
print('(c) Max |J - J_num|:', None if J is None else np.max(np.abs(J - J_num)))
print('(d) J @ 1 =', shift_test)
print('(e) CE gradient =', grad_ce_analytic)


In [ ]:
# Exercise 3: Solution

z = np.array([2.0, 1.0, 0.5, -1.0])
y = 0
K = len(z)
h = 1e-7

# (a) Softmax
s = softmax(z)

# (b) Analytic Jacobian
J_analytic = np.diag(s) - np.outer(s, s)

# (c) Numerical Jacobian
J_numeric = np.zeros((K, K))
for j in range(K):
    zp, zm = z.copy(), z.copy()
    zp[j] += h; zm[j] -= h
    J_numeric[:, j] = (softmax(zp) - softmax(zm)) / (2*h)

# (d) Shift invariance: J @ 1 = 0
shift_result = J_analytic @ np.ones(K)

# (e) Cross-entropy gradient via chain rule
# dL/dz_i = sum_k (dL/ds_k)(ds_k/dz_i)
# dL/ds_k = -1/s_y if k=y, else 0
# dL/dz_i = -1/s_y * J_yi = -1/s_y * s_y*(delta_iy - s_i) = -(delta_iy - s_i) = s_i - delta_iy
e_y = np.zeros(K); e_y[y] = 1.0
grad_ce = s - e_y

# Verify numerically
def ce_loss(z_in, y_in):
    return -log_softmax(z_in)[y_in]

grad_ce_num = np.zeros(K)
for j in range(K):
    zp, zm = z.copy(), z.copy()
    zp[j]+=h; zm[j]-=h
    grad_ce_num[j] = (ce_loss(zp, y) - ce_loss(zm, y)) / (2*h)

header('Exercise 3: Softmax Jacobian')
print(f'Softmax probabilities: {np.round(s, 4)}')
check_close('J analytic == J numeric', np.max(np.abs(J_analytic - J_numeric)), 0, tol=1e-5)
check_close('J @ 1 = 0 (shift invariance)', np.max(np.abs(shift_result)), 0, tol=1e-12)
check_close('CE gradient = s - e_y (analytic == numeric)', np.max(np.abs(grad_ce - grad_ce_num)), 0, tol=1e-5)

print(f'Cross-entropy gradient: {np.round(grad_ce, 4)}')
print(f'  True class (y={y}): {grad_ce[y]:.4f} = s[y]-1 (negative, pushes logit UP)')
print(f'  Wrong classes: {np.round(grad_ce[1:], 4)} = s[j] (positive, pushes logits DOWN)')
print('\nTakeaway: cross-entropy + softmax has the elegant gradient s - e_y because '
      'log cancels exp in the softmax. This is why CE is the universal classification loss.')


---

## Exercise 4 ★★ — Vanishing Gradient Depth Analysis

**Mathematical background:** For a sigmoid network with $L$ layers and unit-norm weight matrices,
the gradient at layer 1 satisfies $\lVert \delta^{[1]} \rVert \le (1/4)^{L-1} \lVert \delta^{[L]} \rVert$.

**Tasks:**
(a) Compute the theoretical upper bound $(1/4)^{L-1}$ for $L = 5, 10, 20, 50$.

(b) Simulate a sigmoid network with orthogonal weight matrices and measure the actual gradient norm at the first layer. Compare to the theoretical bound.

(c) Repeat for tanh. Is the bound tighter or looser?

(d) Show empirically that ReLU does not exhibit this attenuation pattern.


In [ ]:
# Exercise 4: Your Solution

depths = [5, 10, 20, 50]

# (a) Theoretical bounds
sigmoid_bounds = None  # TODO: list of (1/4)**(L-1) for L in depths

print('(a) Theoretical sigmoid bounds:')
for L, b in zip(depths, sigmoid_bounds or []):
    print(f'  L={L}: {b:.2e}')

# (b)-(d): Simulation
n = 64
n_trials = 20

def simulate_gradient_norm(activation, L, n=64, n_trials=20):
    # YOUR CODE HERE
    pass

# Run simulation
for act_name in ['Sigmoid', 'Tanh', 'ReLU']:
    print(f'\n{act_name} gradient norms:')
    for L in [5, 10, 20]:
        # norm = simulate_gradient_norm(...)
        print(f'  L={L}: gradient norm = None')


In [ ]:
# Exercise 4: Solution

np.random.seed(42)
depths = [5, 10, 20]
n = 64
n_trials = 30

# (a) Theoretical bounds
sigmoid_bounds = [(1/4)**(L-1) for L in depths]

def simulate_gradient_norm(activation_fn, L, n=64, n_trials=30):
    """Measure mean gradient norm at layer 1 with orthogonal weight matrices."""
    grad_norms = []
    h = 1e-6
    for _ in range(n_trials):
        # Random orthogonal weight matrices
        Ws = [np.linalg.qr(np.random.randn(n,n))[0] for _ in range(L)]
        # Forward pass
        x = np.random.randn(n) / np.sqrt(n)
        acts = [x]
        for W in Ws:
            pre = W @ acts[-1]
            acts.append(activation_fn(pre))
        # Backward pass
        delta = np.ones(n) / np.sqrt(n)
        for l in range(L-1, 0, -1):
            deriv = (activation_fn(acts[l]+h) - activation_fn(acts[l]-h))/(2*h)
            delta = (Ws[l].T @ delta) * deriv
        grad_norms.append(np.linalg.norm(delta))
    return np.mean(grad_norms)

act_fns = [('Sigmoid', sigmoid), ('Tanh', np.tanh), ('ReLU', relu)]

header('Exercise 4: Vanishing Gradient Depth Analysis')
print(f'{"Depth":<8}', end='')
print(' '.join(f'{name:<14}' for name, _ in act_fns))
print(f'{"":8}', end='')
print(' '.join(f'{"(bound)":<14}' if name == 'Sigmoid' else f'{"":<14}' for name, _ in act_fns))

for L, bound in zip(depths, sigmoid_bounds):
    print(f'{L:<8}', end='')
    for i, (name, fn) in enumerate(act_fns):
        gnorm = simulate_gradient_norm(fn, L)
        if name == 'Sigmoid':
            print(f'{gnorm:.2e}({bound:.2e})', end='  ')
        else:
            print(f'{gnorm:<14.2e}', end='')
    print()

check_true('Sigmoid gradient at L=10 < 1e-4',
           simulate_gradient_norm(sigmoid, 10) < 1e-4)
check_true('ReLU gradient at L=10 > 0.01',
           simulate_gradient_norm(relu, 10) > 0.01)

print('\nTakeaway: sigmoid suffers exponential gradient decay (\"vanishing gradient\'\'); '
      'ReLU and GELU maintain O(1) gradients, enabling training of very deep networks.')


---

## Exercise 5 ★★ — SELU Self-Normalising Property

**Mathematical background:** SELU is the unique scaling of ELU such that if $z \sim \mathcal{N}(0,1)$, then $\operatorname{SELU}(z)$ also has mean 0 and variance 1.

**Tasks:**
(a) Using Monte Carlo with $N = 10^6$ samples, verify that $\mathbb{E}[\operatorname{SELU}(z)] \approx 0$ and $\operatorname{Var}[\operatorname{SELU}(z)] \approx 1$ for $z \sim \mathcal{N}(0,1)$.

(b) Verify that this property does NOT hold for plain ELU ($\lambda = 1$). Compute the mean and variance.

(c) Show that ReLU is NOT self-normalising: compute $\mathbb{E}[\operatorname{ReLU}(z)]$ and $\operatorname{Var}[\operatorname{ReLU}(z)]$ for $z \sim \mathcal{N}(0,1)$. What are the exact expected values (hint: use the half-normal distribution)?

(d) Verify: for ReLU with $z \sim \mathcal{N}(0,1)$, $\mathbb{E}[\operatorname{ReLU}(z)] = 1/\sqrt{2\pi}$ and $\operatorname{Var}[\operatorname{ReLU}(z)] = 1/2 - 1/(2\pi)$.


In [ ]:
# Exercise 5: Your Solution

np.random.seed(42)
N = 1_000_000
z = np.random.normal(0, 1, N)

# (a) SELU statistics
selu_out = None  # TODO: selu(z)
selu_mean = None
selu_var  = None

# (b) ELU statistics (no SELU scaling)
elu_out  = None  # TODO: elu(z) with default alpha=1
elu_mean = None
elu_var  = None

# (c)-(d) ReLU statistics
relu_out  = None  # TODO
relu_mean = None
relu_var  = None

# Exact values for ReLU
relu_mean_exact = None  # TODO: 1/sqrt(2*pi)
relu_var_exact  = None  # TODO: 1/2 - 1/(2*pi)

print('(a) SELU: mean =', selu_mean, ', var =', selu_var)
print('(b) ELU:  mean =', elu_mean,  ', var =', elu_var)
print('(c) ReLU: mean =', relu_mean, ', var =', relu_var)
print('(d) Exact: mean =', relu_mean_exact, ', var =', relu_var_exact)


In [ ]:
# Exercise 5: Solution

np.random.seed(42)
N = 1_000_000
z = np.random.normal(0, 1, N)

# (a) SELU statistics
selu_out  = selu(z)
selu_mean = selu_out.mean()
selu_var  = selu_out.var()

# (b) ELU statistics (no SELU scaling, lambda=1)
elu_out  = elu(z, a=1.0)
elu_mean = elu_out.mean()
elu_var  = elu_out.var()

# (c)-(d) ReLU statistics
relu_out  = relu(z)
relu_mean_mc = relu_out.mean()
relu_var_mc  = relu_out.var()

# Exact: E[max(0,Z)] = integral_0^inf z*phi(z)dz = 1/sqrt(2pi)
relu_mean_exact = 1.0 / np.sqrt(2 * np.pi)
# Var[max(0,Z)] = E[max(0,Z)^2] - E[max(0,Z)]^2
#               = E[Z^2 * 1{Z>0}] - (1/sqrt(2pi))^2
#               = 1/2 - 1/(2*pi)
relu_var_exact = 0.5 - 1.0 / (2 * np.pi)

header('Exercise 5: SELU Self-Normalising Property')
print(f'SELU: mean={selu_mean:.4f} (expected ≈ 0), var={selu_var:.4f} (expected ≈ 1)')
print(f'ELU:  mean={elu_mean:.4f} (NOT zero), var={elu_var:.4f} (NOT 1)')
print(f'ReLU: mean={relu_mean_mc:.4f}, var={relu_var_mc:.4f}')
print(f'ReLU exact: mean={relu_mean_exact:.4f}, var={relu_var_exact:.4f}')

check_true('SELU mean ≈ 0', abs(selu_mean) < 0.01)
check_true('SELU var ≈ 1', abs(selu_var - 1.0) < 0.01)
check_true('ELU mean ≠ 0 (not self-normalising)', abs(elu_mean) > 0.05)
check_close('ReLU mean matches exact formula', relu_mean_mc, relu_mean_exact, tol=0.01)
check_close('ReLU var matches exact formula', relu_var_mc, relu_var_exact, tol=0.01)

print('\nTakeaway: SELU is the unique (lambda, alpha) pair that makes N(0,1) a fixed point '
      'of the activation map. LeCun-normal init + SELU = self-normalising networks (no BatchNorm needed).')


---

## Exercise 6 ★★ — Lipschitz Constants

**Mathematical background:** $K = \sup_z |\sigma'(z)|$ is the Lipschitz constant of a differentiable activation.

**Tasks:**
(a) Prove analytically that $K_{\sigma} = 1/4$ for sigmoid.

(b) Compute Lipschitz constants numerically for: sigmoid, tanh, ReLU, ELU, SELU, GELU, SiLU.

(c) For a 3-layer network with all ReLU activations and weight matrices with spectral norm $\lVert W^{[l]} \rVert_2 = s_l$, what is the Lipschitz constant of the whole network?

(d) Verify that GELU has $K > 1$. At which $z$ is the maximum gradient achieved?


In [ ]:
# Exercise 6: Your Solution

# (b) Compute Lipschitz constants numerically
z_fine = np.linspace(-10, 10, 500_000)
h = 1e-6

activations_to_check = [
    ('Sigmoid', sigmoid),
    ('Tanh', np.tanh),
    ('ReLU', relu),
    ('ELU', elu),
    ('SELU', selu),
    ('GELU', gelu),
    ('SiLU', silu),
]

for name, fn in activations_to_check:
    K = None  # TODO: compute sup|f'(z)|
    print(f'{name}: K = {K}')

# (c) Network Lipschitz constant
s1, s2, s3 = 1.5, 1.2, 0.8  # spectral norms
network_K = None  # TODO: product of K_relu * s_l for each layer
print(f'\n(c) Network Lipschitz constant (K_relu=1): {network_K}')

# (d) GELU K > 1
gelu_K_z = None  # TODO: z at which GELU gradient is maximised
print(f'(d) GELU max gradient at z = {gelu_K_z}')


In [ ]:
# Exercise 6: Solution

z_fine = np.linspace(-10, 10, 1_000_000)
h = 1e-7

def lipschitz(fn):
    deriv = (fn(z_fine+h) - fn(z_fine-h)) / (2*h)
    idx = np.argmax(np.abs(deriv))
    return np.max(np.abs(deriv)), z_fine[idx]

activations_to_check = [
    ('Sigmoid', sigmoid, 0.25),
    ('Tanh', np.tanh, 1.0),
    ('ReLU', relu, 1.0),
    ('ELU (a=1)', lambda z: elu(z, 1.0), 1.0),
    ('SELU', selu, SELU_L),
    ('GELU', gelu, None),  # > 1, exact TBD
    ('SiLU', silu, None),  # > 1, exact TBD
]

header('Exercise 6: Lipschitz Constants')
print(f'{"Activation":<16}  {"K (numeric)":<14}  {"Achieved at z":<15}  Expected')
print('-' * 65)
for name, fn, expected in activations_to_check:
    K_num, z_max = lipschitz(fn)
    exp_str = f'{expected:.4f}' if expected is not None else '>1'
    print(f'{name:<16}  {K_num:<14.6f}  {z_max:<15.4f}  {exp_str}')

K_sig, _ = lipschitz(sigmoid)
K_gelu, z_gelu = lipschitz(gelu)
check_close('Sigmoid K = 1/4', K_sig, 0.25, tol=1e-4)
check_true('GELU K > 1', K_gelu > 1.0)

# (c) Network Lipschitz constant with ReLU (K=1)
s1, s2, s3 = 1.5, 1.2, 0.8
K_relu = 1.0
network_K = (K_relu * s1) * (K_relu * s2) * (K_relu * s3)
print(f'\n(c) 3-layer ReLU network Lipschitz constant: K_relu^3 * s1*s2*s3 = {network_K:.4f}')

print('\nTakeaway: Lipschitz constant bounds gradient propagation and model sensitivity. '
      'Spectral normalisation (WGAN) constrains each layer\'s spectral norm to 1, '
      'ensuring the network has Lipschitz constant <= K_activation^L.')


---

## Exercise 9 ★★★ — Temperature Scaling Calibration

**Mathematical background:** For logits $\mathbf{z}$, temperature scaling applies $\hat{p}_k = \operatorname{softmax}(\mathbf{z}/T)_k$. A well-calibrated model has $\mathbb{E}[\hat{p}_{\hat{y}}] \approx \Pr(\hat{y} = y)$. The optimal $T$ minimises the negative log-likelihood on a held-out validation set:
$$T^* = \arg\min_T \frac{1}{N}\sum_{n=1}^N -\log \operatorname{softmax}(\mathbf{z}^{(n)}/T)_{y^{(n)}}$$

**Tasks:**
(a) Simulate a 5-class classifier: generate 500 logit vectors from $\mathcal{N}(0, 3^2)$ and assign labels by argmax (simulating an overconfident model).

(b) Compute Expected Calibration Error (ECE) before temperature scaling. Bin the top probabilities into 10 equal-width bins and compute ECE = $\sum_b \frac{|B_b|}{N} |\operatorname{acc}(B_b) - \operatorname{conf}(B_b)|$.

(c) Find $T^*$ by minimising NLL over a grid $T \in [0.1, 5.0]$ with 200 steps.

(d) Recompute ECE after scaling by $T^*$. Show that ECE decreases.

(e) Plot confidence histograms before and after scaling (optional if matplotlib available).


In [ ]:
# Exercise 9: Your Solution

np.random.seed(0)
N, K = 500, 5
logits = np.random.randn(N, K) * 3.0
labels = logits.argmax(axis=1)  # overconfident: labels = predicted class

def softmax(z):
    e = np.exp(z - z.max(axis=-1, keepdims=True))
    return e / e.sum(axis=-1, keepdims=True)

def ece(probs, labels, n_bins=10):
    """Expected Calibration Error."""
    # TODO: compute top confidence, accuracy, bin them
    return None

def nll(logits, labels, T):
    """Negative log-likelihood for temperature T."""
    return None  # TODO

# (b) ECE before scaling
probs_raw = softmax(logits)
ece_before = ece(probs_raw, labels)

# (c) Find T* via grid search
T_grid = np.linspace(0.1, 5.0, 200)
T_star = None  # TODO

# (d) ECE after scaling
probs_scaled = softmax(logits / T_star) if T_star else None
ece_after = ece(probs_scaled, labels) if probs_scaled is not None else None

print(f'(b) ECE before scaling: {ece_before}')
print(f'(c) Optimal temperature T* = {T_star}')
print(f'(d) ECE after scaling:  {ece_after}')


In [ ]:
# Exercise 9: Solution

import numpy as np
from scipy.special import softmax as sp_softmax

np.random.seed(0)
N, K = 500, 5
logits = np.random.randn(N, K) * 3.0
labels = logits.argmax(axis=1)

def softmax_T(z, T=1.0):
    z = z / T
    e = np.exp(z - z.max(axis=-1, keepdims=True))
    return e / e.sum(axis=-1, keepdims=True)

def ece(probs, labels, n_bins=10):
    confs = probs.max(axis=1)
    preds = probs.argmax(axis=1)
    accs  = (preds == labels).astype(float)
    bins = np.linspace(0, 1, n_bins + 1)
    total = 0.0
    for lo, hi in zip(bins[:-1], bins[1:]):
        mask = (confs >= lo) & (confs < hi)
        if mask.sum() == 0:
            continue
        total += mask.sum() / N * abs(accs[mask].mean() - confs[mask].mean())
    return total

def nll(T):
    p = softmax_T(logits, T)
    return -np.log(p[np.arange(N), labels] + 1e-12).mean()

# (b) ECE before
probs_raw = softmax_T(logits, T=1.0)
ece_before = ece(probs_raw, labels)

# (c) Grid search for T*
T_grid = np.linspace(0.1, 5.0, 400)
nlls = np.array([nll(T) for T in T_grid])
T_star = T_grid[nlls.argmin()]

# (d) ECE after
probs_scaled = softmax_T(logits, T_star)
ece_after = ece(probs_scaled, labels)

def header(title):
    print('\n' + '=' * len(title))
    print(title)
    print('=' * len(title))

def check_true(name, cond):
    print(f"{'PASS' if cond else 'FAIL'} — {name}")
    return cond

header('Exercise 9: Temperature Scaling Calibration')
print(f'ECE before scaling: {ece_before:.4f}')
print(f'Optimal T*:         {T_star:.4f}')
print(f'ECE after scaling:  {ece_after:.4f}')
check_true('T* in (1, 4) — overconfident model needs T > 1', 1.0 < T_star < 4.0)
check_true('ECE improves after temperature scaling', ece_after < ece_before)

print(f'\nMean confidence before: {probs_raw.max(axis=1).mean():.4f}')
print(f'Mean confidence after:  {probs_scaled.max(axis=1).mean():.4f}')

print('\nTakeaway: Overconfident models (logit variance >> 1) need T > 1 to lower '
      'confidence. Temperature scaling is the cheapest post-hoc calibration method '
      'and is standard practice before deploying classifiers in production.')


---

## Exercise 10 ★★★ — Activation Engineering Toy Model

**Mathematical background:** Mechanistic interpretability uses *activation patching* to identify which neurons causally matter. For a 2-layer MLP $f(\mathbf{x}) = W_2 \sigma(W_1 \mathbf{x} + \mathbf{b}_1) + \mathbf{b}_2$, patching the hidden activation from a 'source' input into the forward pass of a 'target' input tells us which neurons encode the difference between the two inputs.

**Tasks:**
(a) Train a tiny 2-layer MLP with GELU to classify points inside/outside the unit circle on $\mathbb{R}^2$ (binary classification). Use a fixed random weight initialisation (no optimizer — just verify the random init gives some meaningful hidden activations).

(b) Pick two test points: $\mathbf{x}_{\text{in}} = (0.3, 0.3)$ (inside) and $\mathbf{x}_{\text{out}} = (0.9, 0.9)$ (outside). Compute the hidden layer activations for both.

(c) Perform activation patching: replace hidden neuron $k$ in the target forward pass with the source value, and record how the output logit changes. Which neurons have the largest causal effect?

(d) Compute the mean activation value over 1000 random in-distribution points. What fraction of neurons are 'dead' (mean activation < 0.01) for GELU?

(e) Repeat (d) for ReLU. Compare the dead neuron fractions.


In [ ]:
# Exercise 10: Your Solution

import numpy as np

def gelu(z):
    from scipy.special import erf
    return 0.5 * z * (1 + erf(z / np.sqrt(2)))

def relu(z):
    return np.maximum(0, z)

np.random.seed(42)
d_in, d_h = 2, 16

# Fixed random weights
W1 = np.random.randn(d_h, d_in) * np.sqrt(2 / d_in)  # He init
b1 = np.zeros(d_h)
W2 = np.random.randn(1, d_h) * np.sqrt(2 / d_h)
b2 = np.zeros(1)

def forward(x, activation=gelu):
    h = activation(W1 @ x + b1)
    return W2 @ h + b2, h

x_in  = np.array([0.3, 0.3])
x_out = np.array([0.9, 0.9])

# (b) Hidden activations
logit_in,  h_in  = forward(x_in)
logit_out, h_out = forward(x_out)

# (c) Activation patching
patch_effects = None  # TODO: for each neuron k, patch h_out[k] <- h_in[k] and measure delta logit

# (d) Dead neurons for GELU
X_rand = np.random.randn(1000, 2)  # random 2D points
mean_acts_gelu = None  # TODO: mean over 1000 points of gelu hidden activations
dead_gelu = None       # TODO: fraction with mean < 0.01

# (e) Dead neurons for ReLU
mean_acts_relu = None
dead_relu = None

print('(b) Logit for x_in:', logit_in, '  Logit for x_out:', logit_out)
print('(c) Patch effects:', patch_effects)
print('(d) GELU dead neuron fraction:', dead_gelu)
print('(e) ReLU dead neuron fraction:', dead_relu)


In [ ]:
# Exercise 10: Solution

import numpy as np
from scipy.special import erf

def gelu(z): return 0.5 * z * (1 + erf(z / np.sqrt(2)))
def relu(z): return np.maximum(0, z)
def sigmoid(z): return np.where(z>=0, 1/(1+np.exp(-z)), np.exp(z)/(1+np.exp(z)))

np.random.seed(42)
d_in, d_h = 2, 16

W1 = np.random.randn(d_h, d_in) * np.sqrt(2 / d_in)
b1 = np.zeros(d_h)
W2 = np.random.randn(1, d_h) * np.sqrt(2 / d_h)
b2 = np.zeros(1)

def forward(x, activation=gelu):
    h = activation(W1 @ x + b1)
    return (W2 @ h + b2)[0], h

x_in  = np.array([0.3, 0.3])
x_out = np.array([0.9, 0.9])

# (b) Hidden activations
logit_in,  h_in  = forward(x_in)
logit_out, h_out = forward(x_out)

# (c) Activation patching: patch neuron k from source (x_in) into target (x_out)
base_logit = logit_out
patch_effects = np.zeros(d_h)
for k in range(d_h):
    h_patched = h_out.copy()
    h_patched[k] = h_in[k]
    patched_logit = (W2 @ h_patched + b2)[0]
    patch_effects[k] = patched_logit - base_logit

# (d) Dead neurons for GELU
X_rand = np.random.randn(1000, 2)
H_gelu = np.array([gelu(W1 @ x + b1) for x in X_rand])  # shape (1000, d_h)
mean_acts_gelu = H_gelu.mean(axis=0)
dead_gelu = (mean_acts_gelu < 0.01).mean()

# (e) Dead neurons for ReLU
H_relu = np.array([relu(W1 @ x + b1) for x in X_rand])
mean_acts_relu = H_relu.mean(axis=0)
dead_relu = (mean_acts_relu < 0.01).mean()

def header(title):
    print('\n' + '=' * len(title))
    print(title)
    print('=' * len(title))

def check_true(name, cond):
    print(f"{'PASS' if cond else 'FAIL'} — {name}")
    return cond

header('Exercise 10: Activation Engineering Toy Model')
print(f'Logit for x_in  (0.3,0.3): {logit_in:.4f}')
print(f'Logit for x_out (0.9,0.9): {logit_out:.4f}')

top_k = np.argsort(np.abs(patch_effects))[::-1][:3]
print(f'\nTop-3 causal neurons (by |patch effect|):')
for k in top_k:
    print(f'  Neuron {k:2d}: patch effect = {patch_effects[k]:+.4f}')

print(f'\nGELU dead neuron fraction: {dead_gelu:.2%}')
print(f'ReLU dead neuron fraction: {dead_relu:.2%}')

check_true('GELU has fewer dead neurons than ReLU', dead_gelu <= dead_relu)
check_true('At least one neuron has large patch effect (>0.1)', np.abs(patch_effects).max() > 0.1)

print('\nTakeaway: Activation patching localises causal information flow. '
      'GELU is never exactly zero (soft gate), so it has fewer dead neurons '
      'than ReLU — consistent with why transformers use GELU/SwiGLU over ReLU.')


---

## What to Review After Finishing

- [ ] **Exercise 1–2** — Can you derive $\sigma'(z)$ and $\operatorname{GELU}'(z)$ from scratch?
- [ ] **Exercise 3** — Do you understand why the softmax Jacobian is rank-deficient?
- [ ] **Exercise 4** — Can you explain vanishing gradients using the $(1/4)^{L-1}$ bound?
- [ ] **Exercise 5** — Do you know the two SELU constants and what they guarantee?
- [ ] **Exercise 6** — Can you bound a network's Lipschitz constant from spectral norms?
- [ ] **Exercise 7** — Can you derive the He init variance $2/n_\text{in}$ for ReLU?
- [ ] **Exercise 8** — Do you know the SwiGLU hidden dim formula $m = 8d/3$?
- [ ] **Exercise 9** — Do you understand when $T^* > 1$ vs $T^* < 1$ and why?
- [ ] **Exercise 10** — Can you explain what activation patching measures causally?

## References

1. Glorot & Bengio (2010). *Understanding the difficulty of training deep feedforward networks.* AISTATS.
2. He et al. (2015). *Delving deep into rectifiers.* ICCV.
3. Klambauer et al. (2017). *Self-normalizing neural networks.* NeurIPS.
4. Hendrycks & Gimpel (2016). *Gaussian error linear units (GELUs).* arXiv:1606.08415.
5. Guo et al. (2017). *On calibration of modern neural networks.* ICML.
6. Shazeer (2020). *GLU variants improve transformer.* arXiv:2002.05202.
7. Touvron et al. (2023). *LLaMA: Open and efficient foundation language models.* arXiv.
8. Elhage et al. (2022). *Toy models of superposition.* Transformer Circuits Thread.
